# KHUNC

KHUNC squeezes an image into a three-channel latent a quarter of its width and height, then grows it back. This notebook trains one on Imagenette, tests it, and finds out how far the latent can be shrunk before Kodak images start to suffer. Everything lands in `MyDrive/KHUNC/experiments`, one folder per run.

Pick a GPU runtime and run the cells top to bottom.

In [ ]:
import sys

from google.colab import drive

drive.mount("/content/drive")

!test -d /content/KHUNC || git clone -q https://github.com/KadirKess/KHUNC.git /content/KHUNC
!git -C /content/KHUNC pull -q --ff-only
%cd /content/KHUNC
!pip install -q uv && uv pip install -q --python {sys.executable} -e .
!git describe --always --dirty

# The editable install is only picked up after a restart, so point Python at the code directly
sys.path.insert(0, "/content/KHUNC/src")

## Settings

Every knob in one place. `storage` decides what the compressed file holds, `precision` only decides how training does its arithmetic. For integer storage, [`ste`](https://arxiv.org/abs/1308.3432) rounds in the forward pass and lets the gradient through untouched, while [`noise`](https://arxiv.org/abs/1611.01704) adds up to half a quantization step of noise instead. Validation always rounds for real.

`MODE` picks what happens next: train from scratch, fine-tune `CHECKPOINT`, resume a run from its `last.ckpt`, or just evaluate `CHECKPOINT`.

In [ ]:
from khunc import ModelConfig, QuantizationConfig, TrainConfig

config = TrainConfig(
    name="float32-augmented",
    data_dir="data/imagenette2",
    output_dir="/content/drive/MyDrive/KHUNC/experiments",
    model=ModelConfig(
        latent_channels=3,
        base_channels=32,
        encoder_up_layers=2,  # 0 to 4, the decoder gets the rest; anything but 2 means a new model
    ),
    quantization=QuantizationConfig(
        storage="float32",  # float32, float16, bfloat16 or uint8
        bits=8,  # 1 to 8, uint8 only
        training="none",  # none, ste or noise; noise needs uint8
    ),
    epochs=50,
    batch_size=8,
    learning_rate=1e-3,  # 1e-4 is a better start for fine-tuning
    loss="ms_ssim",  # ms_ssim, mse or ms_ssim_mse
    mse_weight=0.1,  # ms_ssim_mse only
    precision="bf16-mixed",  # 32-true, 16-mixed or bf16-mixed; check what the GPU supports
    compile_model=True,
    lr_scheduler_patience=10,
    lr_scheduler_factor=0.1,
    early_stopping_patience=20,  # None turns it off
    image_size=384,  # multiple of 16, above 160 for MS-SSIM
    validation_fraction=0.1,
    seed=42,
    num_workers=2,  # 0 if worker processes misbehave
    augment=True,  # False trains on plain resizes
    crop_scale=(0.5, 1.0),  # fraction of the image area kept before resizing
    horizontal_flip=0.5,  # probability, 0 turns it off
    vertical_flip=0.5,
    color_jitter=0.1,  # brightness, contrast and saturation; 0 turns it off
)

MODE = "train"  # train, fine_tune, resume or evaluate
CHECKPOINT = None  # weights for fine_tune and evaluate, a run's last.ckpt for resume

## Training

Imagenette is downloaded once into `data/`. Its official `val` folder is kept aside as the test set, and validation comes from a seeded slice of `train`. A resumed run reuses the settings saved with it, except for `epochs`, which is the total to reach rather than extra epochs.

In [ ]:
from dataclasses import replace
from pathlib import Path

from khunc import Experiment, download_imagenette, load_model, train

download_imagenette("data")

if MODE == "evaluate":
    model = load_model(CHECKPOINT)
    config = replace(config, model=model.config)
    experiment = Experiment.create(f"{config.name}-evaluation", config.output_dir, config)
else:
    if MODE == "resume":
        previous = Experiment(Path(CHECKPOINT).parents[1]).load_config()
        config = replace(previous, epochs=config.epochs, output_dir=config.output_dir)
    result = train(
        config,
        checkpoint=CHECKPOINT if MODE == "fine_tune" else None,
        resume_from=CHECKPOINT if MODE == "resume" else None,
    )
    model, experiment = result.model, result.experiment

print("Saved to", experiment.path)

In [ ]:
from khunc.plotting import plot_training

metrics = experiment.path / "logs/training/metrics.csv"
if metrics.exists():
    plot_training(metrics, path=experiment.path / "plots/training.png")

## Imagenette test set

In [ ]:
import json

from khunc import test_model

test_metrics = test_model(model, config)
(experiment.path / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2) + "\n")
test_metrics

## How small can the latent get?

Kodak is the final exam: 24 photos at their native resolution, scored with SSIMULACRA2 (higher is better). Colab doesn't ship the metric, so the next cell builds it from libjxl. That takes a few minutes the first time on a runtime.

In [ ]:
%%bash
if [ ! -x /content/libjxl/build/tools/ssimulacra2 ]; then
  apt-get update -qq
  apt-get install -y -qq build-essential cmake pkg-config libbrotli-dev libjpeg-dev libpng-dev
  git clone -q --depth 1 --branch v0.12.0 --recursive --shallow-submodules \
    https://github.com/libjxl/libjxl.git /content/libjxl
  cmake -S /content/libjxl -B /content/libjxl/build \
    -DCMAKE_BUILD_TYPE=Release -DBUILD_TESTING=OFF -DJPEGXL_ENABLE_DEVTOOLS=ON
  cmake --build /content/libjxl/build --target ssimulacra2 -j 2
fi

The same weights go through every storage mode below, so the only thing changing is what gets written to disk. Delete lines to skip modes; integer storage accepts 1 to 8 bits.

In [ ]:
from khunc import QuantizationConfig, compare_quantization, download_kodak
from khunc.plotting import plot_quantization

SSIMULACRA2 = "/content/libjxl/build/tools/ssimulacra2"

kodak = download_kodak("data/kodak")
storage_modes = [
    QuantizationConfig(storage="float32"),
    QuantizationConfig(storage="float16"),
    QuantizationConfig(storage="bfloat16"),
    *[QuantizationConfig(storage="uint8", bits=bits) for bits in [8, 7, 6, 5, 4]],
]
comparison = compare_quantization(
    model, kodak, storage_modes, experiment=experiment, repeats=3, warmup=2, binary=SSIMULACRA2
)

for mode, summary in comparison.summary.items():
    print(
        f"{mode:>10}: SSIMULACRA2 {summary['mean_ssimulacra2']:6.2f}, "
        f"{summary['mean_compressed_bytes'] / 1024:7.2f} KiB, "
        f"compress {summary['mean_encode_ms']:.2f} ms, "
        f"decompress {summary['mean_decode_ms']:.2f} ms"
    )
plot_quantization(comparison, path=experiment.path / "plots/storage-comparison.png")

## Look at one

Numbers are nice, eyes are better. This round-trips one Kodak image (`kodim01` to `kodim24`) through a real `.khunc` file using the storage from the settings.

In [ ]:
from khunc import Codec, inspect_compressed, load_image, save_image
from khunc.plotting import plot_reconstruction

image = "kodim01"

codec = Codec(model, config.quantization)
compressed = experiment.path / f"images/{image}.khunc"
compressed.write_bytes(codec.encode(kodak / f"{image}.png"))
reconstructed = codec.decode(compressed)
save_image(reconstructed, experiment.path / f"images/{image}-reconstructed.png")
plot_reconstruction(
    load_image(kodak / f"{image}.png"),
    reconstructed,
    path=experiment.path / f"plots/{image}.png",
)
inspect_compressed(compressed)

## Your own images

Point `extra_images` at a folder or a list of paths and they get the same treatment as Kodak.

In [ ]:
from khunc import evaluate

extra_images = None

if extra_images is not None:
    extra = evaluate(
        model,
        extra_images,
        quantization=config.quantization,
        experiment=experiment,
        binary=SSIMULACRA2,
    )
    print(extra.summary)